In [1]:
!pip install --quiet "elasticsearch>=8.0.0" sentence-transformers pypdf
!pip install -q pymupdf pillow


# Extract images from PDF documents → save to disk

In [2]:
# =============================================================================
# PDF (scanned) → render page @ high DPI → Nova Lite bbox → crop photos → save
# Resume-safe: skips pages already processed (files already exist)
# Robust: handles Nova returning dict OR list bbox formats; strips code fences; never hard-crashes
# =============================================================================

!pip install -q pymupdf opencv-python-headless pillow boto3

import os, glob, json, re
from typing import Optional, List, Dict, Any, Tuple, Union

import fitz  # PyMuPDF
import cv2
import numpy as np
import boto3


# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
AWS_REGION = "eu-north-1"
AWS_PROFILE: Optional[str] = None

MODEL_ID = "eu.amazon.nova-lite-v1:0"

PDF_DIR = "/home/ec2-user/rag-project/radio_luxembourg"
OUT_DIR = "/home/ec2-user/rag-project/extracted_images_cropped_nova_resume"

DPI = 400                    # HIGH RES output
MAX_PHOTOS_PER_PAGE = 2      # your example
MIN_BOX_AREA_FRAC = 0.06     # ignore tiny boxes
MAX_BOX_AREA_FRAC = 0.75     # ignore near-full-page boxes


# -----------------------------------------------------------------------------
# AWS / Bedrock client
# -----------------------------------------------------------------------------
def make_bedrock_client(region: str, profile: Optional[str] = None):
    if profile:
        print(f"[AWS] Using boto3 profile='{profile}' region='{region}'")
        session = boto3.Session(profile_name=profile, region_name=region)
        return session.client("bedrock-runtime")
    print(f"[AWS] Using default AWS credentials region='{region}'")
    return boto3.client("bedrock-runtime", region_name=region)

brt = make_bedrock_client(AWS_REGION, AWS_PROFILE)
print(f"[AWS] Bedrock client ready. MODEL_ID='{MODEL_ID}'")

os.makedirs(OUT_DIR, exist_ok=True)
print(f"[FS] Output dir: {OUT_DIR}")


# -----------------------------------------------------------------------------
# Render helpers
# -----------------------------------------------------------------------------
def render_page_to_rgb(page: fitz.Page, dpi: int = 400) -> np.ndarray:
    zoom = dpi / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4:
        rgb = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    else:
        # Usually already RGB; keep guard
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if pix.n == 3 else img
    return rgb

def rgb_to_png_bytes(rgb: np.ndarray) -> bytes:
    ok, buf = cv2.imencode(".png", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    if not ok:
        raise RuntimeError("Failed to encode PNG")
    return buf.tobytes()


# -----------------------------------------------------------------------------
# Robust JSON extraction: handles ```json ... ``` fences and extra text
# -----------------------------------------------------------------------------
def extract_json_object(text: str) -> Dict[str, Any]:
    """
    Try very hard to pull a JSON object from model text.
    Handles:
      - ```json ... ```
      - leading/trailing explanations
      - whitespace / weird chars
    Returns parsed dict, or raises JSONDecodeError if impossible.
    """
    t = text.strip()

    # If fenced code block exists, extract inside the first fence
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", t, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        candidate = fence.group(1).strip()
        return json.loads(candidate)

    # Otherwise, extract between first { and last }
    start = t.find("{")
    end = t.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = t[start:end+1].strip()
        return json.loads(candidate)

    # As a last attempt, try parse the whole thing
    return json.loads(t)


# -----------------------------------------------------------------------------
# Nova call
# -----------------------------------------------------------------------------
def nova_detect_photos_bboxes(rgb_page: np.ndarray) -> List[Any]:
    """
    Returns raw 'photos' from the model.
    We purposely allow Any because model may return:
      - [{"label":"photo","bbox":[...]}]
      - [[x1,y1,x2,y2], ...]
    """
    png_bytes = rgb_to_png_bytes(rgb_page)

    prompt = """
Return ONLY valid JSON (no markdown, no explanations).

Detect ONLY continuous-tone photographic images on this scanned document page.
Do NOT return boxes for text blocks, lists, tables, headings, page numbers, footnotes, borders, margins, or whitespace.
If there are NO photos, return {"photos": []}.
If a photo has a caption under it, include ONLY the photo pixels (exclude caption).

Bounding boxes must be on a 0..1000 integer scale:
bbox = [x1, y1, x2, y2] where (0,0) top-left and (1000,1000) bottom-right.

JSON schema:
{"photos":[{"label":"photo","bbox":[0,0,0,0]}]}
"""

    resp = brt.converse(
        modelId=MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {"image": {"format": "png", "source": {"bytes": png_bytes}}},
                {"text": prompt},
            ],
        }],
        inferenceConfig={"maxTokens": 700, "temperature": 0.0},
    )

    text_out = resp["output"]["message"]["content"][0]["text"]
    data = extract_json_object(text_out)
    photos = data.get("photos", [])
    return photos


# -----------------------------------------------------------------------------
# Post-filters: scale, size filters, text-heavy filter
# -----------------------------------------------------------------------------
def scale_bbox_1000_to_pixels(bbox: List[int], W: int, H: int) -> Optional[Tuple[int,int,int,int]]:
    x1, y1, x2, y2 = bbox
    px1 = int(round(x1 / 1000.0 * W))
    py1 = int(round(y1 / 1000.0 * H))
    px2 = int(round(x2 / 1000.0 * W))
    py2 = int(round(y2 / 1000.0 * H))

    px1 = max(0, min(W - 1, px1))
    py1 = max(0, min(H - 1, py1))
    px2 = max(0, min(W, px2))
    py2 = max(0, min(H, py2))

    if px2 <= px1 or py2 <= py1:
        return None
    return (px1, py1, px2, py2)

def box_area_frac(box_px: Tuple[int,int,int,int], W: int, H: int) -> float:
    x1, y1, x2, y2 = box_px
    return ((x2 - x1) * (y2 - y1)) / float(W * H)

def looks_text_heavy(crop_rgb: np.ndarray) -> bool:
    gray = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th_inv = 255 - th

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    th_inv = cv2.morphologyEx(th_inv, cv2.MORPH_OPEN, kernel, iterations=1)

    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(th_inv, connectivity=8)

    small = 0
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if 8 <= area <= 250:
            small += 1

    return small > 250


# -----------------------------------------------------------------------------
# Helpers for resume + saving
# -----------------------------------------------------------------------------
def page_already_processed(out_dir: str, pdf_base: str, page_num: int) -> bool:
    """
    If ANY output file for that page exists, we consider it processed.
    (You can tighten this to require img01+img02 if you want.)
    """
    pattern = os.path.join(out_dir, f"{pdf_base}_p{page_num:04d}_img*.png")
    return len(glob.glob(pattern)) > 0

def crop_and_save(rgb_page: np.ndarray, boxes_px: List[Tuple[int,int,int,int]],
                  out_dir: str, pdf_base: str, page_num: int) -> int:
    os.makedirs(out_dir, exist_ok=True)
    saved = 0
    boxes_px = sorted(boxes_px, key=lambda b: b[1])  # top->bottom

    for i, (x1, y1, x2, y2) in enumerate(boxes_px, start=1):
        crop = rgb_page[y1:y2, x1:x2]
        out_path = os.path.join(out_dir, f"{pdf_base}_p{page_num:04d}_img{i:02d}.png")
        ok = cv2.imwrite(out_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        if ok:
            saved += 1
            print(f"      [SAVE] {out_path}  (px: {x1},{y1},{x2},{y2})")
        else:
            print(f"      [SAVE][WARN] Failed writing {out_path}")
    return saved


def normalize_photos_to_bbox_list(photos: List[Any]) -> List[List[int]]:
    """
    Convert possible formats into a list of bbox lists [x1,y1,x2,y2].
    Accepts:
      - {"bbox":[...]}
      - [x1,y1,x2,y2]
    """
    bboxes: List[List[int]] = []

    for item in photos:
        # Case 1: dict with bbox
        if isinstance(item, dict) and "bbox" in item:
            bb = item["bbox"]
            if isinstance(bb, list) and len(bb) == 4:
                bboxes.append([int(bb[0]), int(bb[1]), int(bb[2]), int(bb[3])])
            continue

        # Case 2: direct bbox list
        if isinstance(item, list) and len(item) == 4 and all(isinstance(x, (int, float)) for x in item):
            bboxes.append([int(item[0]), int(item[1]), int(item[2]), int(item[3])])
            continue

        # Otherwise ignore quietly
    return bboxes


# -----------------------------------------------------------------------------
# Main runner (resume-safe)
# -----------------------------------------------------------------------------
def extract_photos_with_nova(pdf_dir: str, out_dir: str, dpi: int = 400, max_photos_per_page: int = 2):
    pdf_files = sorted(glob.glob(os.path.join(pdf_dir, "*.pdf")))
    if not pdf_files:
        print(f"[ERROR] No PDFs found in: {pdf_dir}")
        return

    print(f"[START] PDFs={len(pdf_files)} | Input={pdf_dir}")
    print(f"[START] Output={out_dir}")
    print(f"[START] DPI={dpi} | max_photos_per_page={max_photos_per_page}")
    print(f"[START] Filters: min_area={MIN_BOX_AREA_FRAC:.0%} max_area={MAX_BOX_AREA_FRAC:.0%}")

    total_saved = 0
    total_skipped_pages = 0
    total_pages = 0

    for pdf_i, pdf_path in enumerate(pdf_files, start=1):
        pdf_base = os.path.splitext(os.path.basename(pdf_path))[0]
        print("\n" + "="*100)
        print(f"[PDF {pdf_i}/{len(pdf_files)}] {pdf_base}.pdf")
        print(f"[PDF] Path: {pdf_path}")

        doc = fitz.open(pdf_path)
        num_pages = len(doc)
        print(f"[PDF] Pages: {num_pages}")

        for pidx in range(num_pages):
            total_pages += 1
            page_num = pidx + 1

            # Resume: skip if we already saved crops for this page
            if page_already_processed(out_dir, pdf_base, page_num):
                total_skipped_pages += 1
                print(f"  [PAGE {page_num:4d}/{num_pages}] SKIP (already processed: {pdf_base}_p{page_num:04d}_img*.png)")
                continue

            print(f"\n  [PAGE {page_num:4d}/{num_pages}] Render page → RGB (dpi={dpi})")
            try:
                rgb = render_page_to_rgb(doc[pidx], dpi=dpi)
            except Exception as e:
                print(f"  [PAGE][ERROR] Rendering failed: {e}  (skipping page)")
                continue

            H, W = rgb.shape[:2]
            print(f"  [PAGE] Render size: {W}x{H}px")

            # Nova detection (never crash the whole run)
            try:
                print("    [Nova] Detecting photo boxes...")
                photos_raw = nova_detect_photos_bboxes(rgb)
            except Exception as e:
                print(f"    [Nova][ERROR] Detection failed: {e}  (skipping page)")
                continue

            # Normalize photos into list of bbox lists
            bboxes_1000 = normalize_photos_to_bbox_list(photos_raw)
            print(f"    [Nova] Returned {len(bboxes_1000)} bbox(es) after normalization.")

            if not bboxes_1000:
                print("    [PAGE] No photo boxes. Moving on.")
                continue

            # Scale and filter by size
            px_boxes: List[Tuple[int,int,int,int]] = []
            for bb in bboxes_1000:
                scaled = scale_bbox_1000_to_pixels(bb, W, H)
                if not scaled:
                    continue

                frac = box_area_frac(scaled, W, H)
                if frac < MIN_BOX_AREA_FRAC:
                    print(f"      [FILTER] drop tiny box (area {frac:.2%})")
                    continue
                if frac > MAX_BOX_AREA_FRAC:
                    print(f"      [FILTER] drop huge box (area {frac:.2%})")
                    continue

                px_boxes.append(scaled)

            print(f"    [PAGE] Boxes after size filters: {len(px_boxes)}")
            if not px_boxes:
                print("    [PAGE] Nothing left after filters. Moving on.")
                continue

            # Keep biggest N
            px_boxes = sorted(px_boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]), reverse=True)[:max_photos_per_page]
            px_boxes = sorted(px_boxes, key=lambda b: b[1])  # top->bottom
            print(f"    [PAGE] Keeping top {len(px_boxes)} box(es) for crop check.")

            # Text-heavy filter
            final_boxes: List[Tuple[int,int,int,int]] = []
            for b in px_boxes:
                x1,y1,x2,y2 = b
                crop = rgb[y1:y2, x1:x2]
                if looks_text_heavy(crop):
                    print("      [FILTER] crop looks text-heavy -> drop")
                    continue
                final_boxes.append(b)

            print(f"    [PAGE] Boxes after text-heavy filter: {len(final_boxes)}")
            if not final_boxes:
                print("    [PAGE] No valid photo crops. Moving on.")
                continue

            # Save
            saved = crop_and_save(rgb, final_boxes, out_dir, pdf_base, page_num)
            total_saved += saved
            print(f"  [PAGE] Saved {saved} crop(s).")

        doc.close()
        print(f"\n[PDF DONE] {pdf_base}.pdf completed.")

    print("\n" + "="*100)
    print(f"[DONE] Total pages seen: {total_pages}")
    print(f"[DONE] Pages skipped (already processed): {total_skipped_pages}")
    print(f"[DONE] Total crops saved: {total_saved}")
    print(f"[DONE] Output folder: {out_dir}")


# -----------------------------------------------------------------------------
# RUN
# -----------------------------------------------------------------------------
extract_photos_with_nova(PDF_DIR, OUT_DIR, dpi=DPI, max_photos_per_page=MAX_PHOTOS_PER_PAGE)


[AWS] Using default AWS credentials region='eu-north-1'
[AWS] Bedrock client ready. MODEL_ID='eu.amazon.nova-lite-v1:0'
[FS] Output dir: /home/ec2-user/rag-project/extracted_images_cropped_nova_resume
[START] PDFs=10 | Input=/home/ec2-user/rag-project/radio_luxembourg
[START] Output=/home/ec2-user/rag-project/extracted_images_cropped_nova_resume
[START] DPI=400 | max_photos_per_page=2
[START] Filters: min_area=6% max_area=75%

[PDF 1/10] Radio Luxembourg 1933-1993.pdf
[PDF] Path: /home/ec2-user/rag-project/radio_luxembourg/Radio Luxembourg 1933-1993.pdf
[PDF] Pages: 273

  [PAGE    1/273] Render page → RGB (dpi=400)
  [PAGE] Render size: 2373x3606px
    [Nova] Detecting photo boxes...
    [Nova] Returned 0 bbox(es) after normalization.
    [PAGE] No photo boxes. Moving on.

  [PAGE    2/273] Render page → RGB (dpi=400)
  [PAGE] Render size: 2334x3506px
    [Nova] Detecting photo boxes...
    [Nova] Returned 0 bbox(es) after normalization.
    [PAGE] No photo boxes. Moving on.

  [PAGE 

# Text insert

In [ ]:
# ======================= PDF → VECTORS → ELASTIC =====================
# This cell will:
#  - install dependencies (safe to re-run)
#  - connect to Elasticsearch
#  - create an index with a dense_vector field
#  - read all PDFs from PDF_DIR
#  - do paragraph-aware semantic chunking with overlap
#  - create multilingual embeddings on CPU and index them
#  - PRINT PROGRESS: which book, which page, how many chunks, totals

# ---------- install deps (idempotent) ----------------------------------------
!pip install -q "elasticsearch>=8.0.0" sentence-transformers pypdf

# ---------- CONFIG: EDIT THESE VALUES ----------------------------------------

ES_URL       = "http://localhost:9200"        # Elasticsearch as seen from EC2
ES_USER      = "elastic"                      # your ES username
ES_PASSWORD  = "YOUR-PASSWORD"        # <<< CHANGE THIS

INDEX_NAME   = "radio_luxembourg_books"       # index name

# Folder where your PDFs live on the EC2 (inside Jupyter)
PDF_DIR      = "/home/ec2-user/rag-project/radio_luxembourg"

# Chunking parameters
MAX_CHARS      = 1200    # target chunk length (characters)
OVERLAP_CHARS  = 200     # overlapping chars between neighbouring chunks

# Multilingual embedding model (FR/DE/LB), CPU-only
MODEL_NAME   = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
EMB_BATCH    = 8 #16        # embedding batch size (lower = less RAM, slower)

# ---------- IMPORTS -----------------------------------------------------------

import os
import glob
import re
from typing import List, Iterable, Tuple
from time import time

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ---------- CONNECT TO ELASTICSEARCH -----------------------------------------

es = Elasticsearch(ES_URL, basic_auth=(ES_USER, ES_PASSWORD))
info = es.info().body
print("Connected to Elasticsearch cluster:", info.get("cluster_name", info))

# ---------- LOAD EMBEDDING MODEL (CPU) ----------------------------------------

print(f"\nLoading embedding model on CPU: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME, device="cpu")
EMBED_DIMS = model.get_sentence_embedding_dimension()
print("Embedding dimension:", EMBED_DIMS)

# ---------- CREATE INDEX (IF NEEDED) -----------------------------------------

mapping = {
    "mappings": {
        "properties": {
            "book_title":   {"type": "keyword"},
            "file_path":    {"type": "keyword"},   # full path on disk
            "pdf_rel_path": {"type": "keyword"},   # e.g. "radio_luxembourg/Radio Luxembourg 1933-1993.pdf"
            "page_number":  {"type": "integer"},
            "chunk_id":     {"type": "integer"},
            "chunk_text":   {"type": "text"},
            "language":     {"type": "keyword"},   # 'de','fr','lb','multi', ...
            "embedding": {
                "type": "dense_vector",
                "dims": EMBED_DIMS,
                "index": True,
                "similarity": "cosine"
            }
        }
    }
}

if not es.indices.exists(index=INDEX_NAME):
    es.indices.create(index=INDEX_NAME, body=mapping)
    print(f"\nIndex '{INDEX_NAME}' created.")
else:
    print(f"\nIndex '{INDEX_NAME}' already exists – using existing mapping.")

# ---------- SEMANTIC-ISH CHUNKING --------------------------------------------

def split_into_paragraphs(text: str) -> List[str]:
    """Split raw page text into paragraphs separated by blank lines."""
    text = text.replace("\r\n", "\n")
    paras = re.split(r"\n{2,}", text)
    return [p.strip() for p in paras if p.strip()]

def make_semantic_chunks_for_page(
    text: str,
    max_chars: int = MAX_CHARS,
    overlap_chars: int = OVERLAP_CHARS,
) -> List[str]:
    """
    Paragraph-aware chunking within a single page.
    Keeps paragraphs together until max_chars is reached.
    When splitting, creates neighbour overlap for smoother semantics.
    """
    paragraphs = split_into_paragraphs(text)
    raw_chunks: List[str] = []
    current = ""

    for p in paragraphs:
        if not current:
            current = p
            continue

        if len(current) + len(p) + 2 <= max_chars:
            current = current + "\n\n" + p
        else:
            raw_chunks.append(current)

            # Very long paragraph → slice with overlap
            if len(p) > max_chars:
                start = 0
                while start < len(p):
                    end = start + max_chars
                    raw_chunks.append(p[start:end])
                    start = end - overlap_chars
                current = ""
            else:
                current = p

    if current:
        raw_chunks.append(current)

    # Add overlap between neighbouring chunks on the same page
    chunks: List[str] = []
    for i, chunk in enumerate(raw_chunks):
        if i == 0:
            chunks.append(chunk)
        else:
            overlap = raw_chunks[i-1][-overlap_chars:]
            chunks.append(overlap + "\n\n" + chunk)

    return chunks

# ---------- PDF TEXT EXTRACTION (PER PAGE) -----------------------------------

def extract_pages_from_pdf(pdf_path: str) -> List[str]:
    """Return list of page texts (index 0 = page 1)."""
    reader = PdfReader(pdf_path)
    pages = []
    for idx, page in enumerate(reader.pages):
        try:
            pages.append(page.extract_text() or "")
        except Exception as e:
            print(f"    [WARN] Failed to extract text from page {idx+1}: {e}")
            pages.append("")
    return pages

# ---------- SIMPLE LANGUAGE TAG FROM FILENAME --------------------------------

def guess_language_from_filename(filename: str) -> str:
    fn = filename.lower()
    if "deutsch" in fn or "german" in fn or fn.endswith("_de.pdf"):
        return "de"
    if "francais" in fn or "français" in fn or fn.endswith("_fr.pdf"):
        return "fr"
    # extend with your own rules if file names encode language
    return "multi"  # unknown / mixed

# ---------- GENERATOR OF BULK ACTIONS ----------------------------------------

def iter_es_actions() -> Iterable[dict]:
    pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
    total_pdfs = len(pdf_files)

    if not pdf_files:
        print(f"No PDF files found in {PDF_DIR}")
        return

    global_doc_counter = 0

    for file_idx, pdf_path in enumerate(pdf_files):
        file_name = os.path.basename(pdf_path)
        print(f"\n[{file_idx+1}/{total_pdfs}] Processing file: {file_name}")
        print(f"    Path: {pdf_path}")

        pages = extract_pages_from_pdf(pdf_path)
        num_pages = len(pages)
        language = guess_language_from_filename(file_name)
        print(f"    Detected language tag: {language}")
        print(f"    Total pages detected: {num_pages}")

        all_chunks: List[str] = []
        all_meta: List[Tuple[int, int]] = []  # (page_number, chunk_id)
        chunk_counter = 0

        for page_idx, page_text in enumerate(pages):
            page_number = page_idx + 1
            if not page_text.strip():
                print(f"    Page {page_number:3d}/{num_pages}: [NO TEXT] – skipping")
                continue

            page_chunks = make_semantic_chunks_for_page(page_text)
            print(f"    Page {page_number:3d}/{num_pages}: {len(page_chunks)} chunk(s)")

            for ch in page_chunks:
                all_chunks.append(ch)
                all_meta.append((page_number, chunk_counter))
                chunk_counter += 1

        if not all_chunks:
            print("    -> No text extracted from this PDF, skipping.")
            continue

        print(f"    Total chunks for this book: {len(all_chunks)}")
        print("    Encoding embeddings on CPU ...")

        embeddings = model.encode(
            all_chunks,
            normalize_embeddings=True,
            convert_to_numpy=True,
            batch_size=EMB_BATCH,
            show_progress_bar=True,
        )

        for (chunk_text, emb_vec), (page_number, chunk_id) in zip(
            zip(all_chunks, embeddings), all_meta
        ):
            pdf_rel_path = f"radio_luxembourg/{file_name}"  # for Jupyter 'files' links
            global_doc_counter += 1

            # Print a light progress line every 100 docs
            if global_doc_counter % 100 == 0:
                print(f"      -> Prepared {global_doc_counter} chunks so far "
                      f"(book: {file_name}, page: {page_number}, chunk_id: {chunk_id})")

            yield {
                "_index": INDEX_NAME,
                "_id": f"{file_name}-p{page_number}-c{chunk_id}",
                "_source": {
                    "book_title":   file_name,
                    "file_path":    pdf_path,
                    "pdf_rel_path": pdf_rel_path,
                    "page_number":  page_number,
                    "chunk_id":     chunk_id,
                    "chunk_text":   chunk_text,
                    "language":     language,
                    "embedding":    emb_vec.tolist(),
                },
            }

    print("\nFinished preparing all chunks for all PDFs.")

# ---------- RUN BULK INDEXING ------------------------------------------------

print("\nStarting bulk indexing into Elasticsearch...")
t0 = time()
success_count, errors = bulk(
    es,
    iter_es_actions(),
    chunk_size=100,
    request_timeout=300,
    raise_on_error=False,
)
t1 = time()

print(f"\nDONE. Successfully indexed documents: {success_count}")
print(f"Took {(t1 - t0):.1f} seconds total.")
if errors:
    print("Some errors occurred (showing first 3):")
    for e in errors[:3]:
        print(e)

Connected to Elasticsearch cluster: elasticsearch

Loading embedding model on CPU: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Embedding dimension: 768

Index 'radio_luxembourg_books' created.

Starting bulk indexing into Elasticsearch...

[1/10] Processing file: Radio Luxembourg 1933-1993.pdf
    Path: /home/ec2-user/rag-project/radio_luxembourg/Radio Luxembourg 1933-1993.pdf
    Detected language tag: multi
    Total pages detected: 273
    Page   1/273: 1 chunk(s)
    Page   2/273: [NO TEXT] – skipping
    Page   3/273: 1 chunk(s)
    Page   4/273: [NO TEXT] – skipping
    Page   5/273: 1 chunk(s)
    Page   6/273: 1 chunk(s)
    Page   7/273: 1 chunk(s)
    Page   8/273: 1 chunk(s)
    Page   9/273: 1 chunk(s)
    Page  10/273: 1 chunk(s)
    Page  11/273: 1 chunk(s)
    Page  12/273: 1 chunk(s)
    Page  13/273: 1 chunk(s)
    Page  14/273: 1 chunk(s)
    Page  15/273: 1 chunk(s)
    Page  16/273: 1 chunk(s)
    Page  17/273: 1 chunk(s)
    Page  18/273: [NO TEXT] 

Batches: 100%|██████████| 33/33 [00:28<00:00,  1.15it/s]
/tmp/ipykernel_111035/2836073616.py:257: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  success_count, errors = bulk(


      -> Prepared 100 chunks so far (book: Radio Luxembourg 1933-1993.pdf, page: 107, chunk_id: 99)
      -> Prepared 200 chunks so far (book: Radio Luxembourg 1933-1993.pdf, page: 211, chunk_id: 199)

[2/10] Processing file: Radio Luxembourg 1979.pdf
    Path: /home/ec2-user/rag-project/radio_luxembourg/Radio Luxembourg 1979.pdf
    Detected language tag: multi
    Total pages detected: 98
    Page   1/98: 1 chunk(s)
    Page   2/98: 1 chunk(s)
    Page   3/98: 1 chunk(s)
    Page   4/98: 1 chunk(s)
    Page   5/98: 1 chunk(s)
    Page   6/98: 1 chunk(s)
    Page   7/98: 1 chunk(s)
    Page   8/98: 1 chunk(s)
    Page   9/98: 1 chunk(s)
    Page  10/98: 1 chunk(s)
    Page  11/98: 1 chunk(s)
    Page  12/98: 1 chunk(s)
    Page  13/98: 1 chunk(s)
    Page  14/98: 1 chunk(s)
    Page  15/98: 1 chunk(s)
    Page  16/98: 1 chunk(s)
    Page  17/98: 1 chunk(s)
    Page  18/98: 1 chunk(s)
    Page  19/98: 1 chunk(s)
    Page  20/98: 1 chunk(s)
    Page  21/98: 1 chunk(s)
    Page  22/98: 1

Batches: 100%|██████████| 13/13 [00:09<00:00,  1.30it/s]


      -> Prepared 300 chunks so far (book: Radio Luxembourg 1979.pdf, page: 41, chunk_id: 40)

[3/10] Processing file: Radio Luxembourg Record Stars Book No5.pdf
    Path: /home/ec2-user/rag-project/radio_luxembourg/Radio Luxembourg Record Stars Book No5.pdf
    Detected language tag: multi
    Total pages detected: 145
    Page   1/145: 1 chunk(s)
    Page   2/145: [NO TEXT] – skipping
    Page   3/145: 1 chunk(s)
    Page   4/145: 1 chunk(s)
    Page   5/145: 1 chunk(s)
    Page   6/145: 1 chunk(s)
    Page   7/145: 1 chunk(s)
    Page   8/145: 1 chunk(s)
    Page   9/145: 1 chunk(s)
    Page  10/145: 1 chunk(s)
    Page  11/145: 1 chunk(s)
    Page  12/145: 1 chunk(s)
    Page  13/145: [NO TEXT] – skipping
    Page  14/145: 1 chunk(s)
    Page  15/145: 1 chunk(s)
    Page  16/145: 1 chunk(s)
    Page  17/145: 1 chunk(s)
    Page  18/145: 1 chunk(s)
    Page  19/145: 1 chunk(s)
    Page  20/145: 1 chunk(s)
    Page  21/145: 1 chunk(s)
    Page  22/145: 1 chunk(s)
    Page  23/145: 1 

...

      -> Prepared 1200 chunks so far (book: The Radio Luxembourg Story.pdf, page: 98, chunk_id: 94)
      -> Prepared 1300 chunks so far (book: The Radio Luxembourg Story.pdf, page: 198, chunk_id: 194)

[9/10] Processing file: The station of the stars - Radio Luxembourg - 1965.pdf
    Path: /home/ec2-user/rag-project/radio_luxembourg/The station of the stars - Radio Luxembourg - 1965.pdf
    Detected language tag: multi
    Total pages detected: 20
    Page   1/20: 1 chunk(s)
    Page   2/20: [NO TEXT] – skipping
    Page   3/20: 1 chunk(s)
    Page   4/20: 1 chunk(s)
    Page   5/20: 1 chunk(s)
    Page   6/20: 1 chunk(s)
    Page   7/20: 1 chunk(s)
    Page   8/20: 1 chunk(s)
    Page   9/20: 1 chunk(s)
    Page  10/20: 1 chunk(s)
    Page  11/20: 1 chunk(s)
    Page  12/20: 1 chunk(s)
    Page  13/20: 1 chunk(s)
    Page  14/20: 1 chunk(s)
    Page  15/20: 1 chunk(s)
    Page  16/20: 1 chunk(s)
    Page  17/20: 1 chunk(s)
    Page  18/20: 1 chunk(s)
    Page  19/20: 1 chunk(s)
    P

Batches: 100%|██████████| 3/3 [00:01<00:00,  1.51it/s]



[10/10] Processing file: The transmitters and studios of Radio Luxembourg.pdf
    Path: /home/ec2-user/rag-project/radio_luxembourg/The transmitters and studios of Radio Luxembourg.pdf
    Detected language tag: multi
    Total pages detected: 16
    Page   1/16: 1 chunk(s)
    Page   2/16: 1 chunk(s)
    Page   3/16: 1 chunk(s)
    Page   4/16: 1 chunk(s)
    Page   5/16: 1 chunk(s)
    Page   6/16: 1 chunk(s)
    Page   7/16: 1 chunk(s)
    Page   8/16: 1 chunk(s)
    Page   9/16: 1 chunk(s)
    Page  10/16: 1 chunk(s)
    Page  11/16: 1 chunk(s)
    Page  12/16: 1 chunk(s)
    Page  13/16: 1 chunk(s)
    Page  14/16: 1 chunk(s)
    Page  15/16: 1 chunk(s)
    Page  16/16: 1 chunk(s)
    Total chunks for this book: 16
    Encoding embeddings on CPU ...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.17it/s]


Finished preparing all chunks for all PDFs.

DONE. Successfully indexed documents: 1370
Took 161.6 seconds total.
